# 03 - Silver to Base Gold

Builds business-ready base KPI tables and an interim advisory-only agent context from conformed Silver facts. Notebook 05 replaces the interim context with the governed superset.

Reruns use `CREATE OR REPLACE TABLE` and do not duplicate data.

In [ ]:
from pyspark.sql import functions as F

# Headline flight, turnaround, passenger-flow, energy, maintenance, and incident KPIs per airport.
spark.sql('''
CREATE OR REPLACE TABLE gold_kpi_daily_summary AS
WITH f AS (
  SELECT airport_id, COUNT(*) AS flights,
         AVG(turnaround_minutes) AS avg_turnaround_min,
         AVG(on_time_arrival_flag) * 100 AS on_time_arrival_rate,
         AVG(on_time_flag) * 100 AS on_time_departure_rate,
         AVG(arrival_delay_minutes) AS avg_arrival_delay_min,
         AVG(departure_delay_minutes) AS avg_departure_delay_min,
         SUM(passenger_count) AS total_pax
  FROM fact_flight_turnaround_events GROUP BY airport_id),
q AS (SELECT airport_id, AVG(wait_time_min) AS avg_queue_wait_min FROM fact_passenger_queue_metrics GROUP BY airport_id),
e AS (SELECT airport_id, SUM(kwh) AS total_kwh FROM fact_energy_metering GROUP BY airport_id),
m AS (SELECT airport_id, SUM(CASE WHEN anomaly_flag THEN 1 ELSE 0 END) AS maintenance_anomaly_count FROM fact_maintenance_events GROUP BY airport_id),
i AS (SELECT airport_id, COUNT(*) AS incident_count FROM fact_operational_incidents GROUP BY airport_id)
SELECT f.airport_id, f.flights,
       ROUND(f.on_time_arrival_rate,1) AS on_time_arrival_rate,
       ROUND(f.on_time_departure_rate,1) AS on_time_departure_rate,
       ROUND(f.avg_arrival_delay_min,1) AS avg_arrival_delay_min,
       ROUND(f.avg_departure_delay_min,1) AS avg_departure_delay_min,
       ROUND(f.avg_turnaround_min,1) AS avg_turnaround_min,
       ROUND(q.avg_queue_wait_min,1) AS avg_queue_wait_min,
       COALESCE(m.maintenance_anomaly_count,0) AS maintenance_anomaly_count,
       ROUND(e.total_kwh / f.flights,1) AS energy_kwh_per_flight,
       ROUND(e.total_kwh / f.total_pax,2) AS energy_kwh_per_pax,
       COALESCE(i.incident_count,0) AS incident_count,
       true AS is_synthetic,
       'Derived analytical data' AS data_classification
FROM f LEFT JOIN q ON f.airport_id=q.airport_id
LEFT JOIN e ON f.airport_id=e.airport_id
LEFT JOIN m ON f.airport_id=m.airport_id
LEFT JOIN i ON f.airport_id=i.airport_id
''')
print('gold_kpi_daily_summary built')

In [ ]:
# --- gold_turnaround_by_hour: turnaround + on-time trend ---
spark.sql('''
CREATE OR REPLACE TABLE gold_turnaround_by_hour AS
SELECT airport_id, event_hour,
       COUNT(*) AS flights,
       ROUND(AVG(turnaround_minutes), 1) AS avg_turnaround_min,
       ROUND(AVG(on_time_flag) * 100, 1) AS on_time_rate
FROM fact_flight_turnaround_events
GROUP BY airport_id, event_hour
''')
print('gold_turnaround_by_hour built')

In [ ]:
# --- gold_queue_by_hour: passenger flow trend ---
spark.sql('''
CREATE OR REPLACE TABLE gold_queue_by_hour AS
SELECT airport_id, checkpoint, event_hour,
       ROUND(AVG(wait_time_min), 1) AS avg_wait_min,
       ROUND(AVG(queue_length), 1) AS avg_queue_length,
       SUM(throughput_pax) AS throughput_pax
FROM fact_passenger_queue_metrics
GROUP BY airport_id, checkpoint, event_hour
''')
print('gold_queue_by_hour built')

In [ ]:
# --- gold_gate_utilization: occupancy per gate ---
spark.sql('''
CREATE OR REPLACE TABLE gold_gate_utilization AS
SELECT g.airport_id, g.gate_id,
       COUNT(f.flight_event_id) AS flights,
       ROUND(COALESCE(SUM((unix_timestamp(f.actual_departure) - unix_timestamp(f.actual_arrival)) / 60), 0), 0) AS occupied_minutes,
       ROUND(COALESCE(SUM((unix_timestamp(f.actual_departure) - unix_timestamp(f.actual_arrival)) / 60), 0) / (24 * 60) * 100, 1) AS utilization_pct
FROM dim_gate g
LEFT JOIN fact_flight_turnaround_events f ON g.gate_id = f.gate_id
GROUP BY g.airport_id, g.gate_id
''')
print('gold_gate_utilization built')

In [ ]:
# --- gold_energy_summary: energy per airport / hour ---
spark.sql('''
CREATE OR REPLACE TABLE gold_energy_summary AS
SELECT airport_id, event_hour, ROUND(SUM(kwh), 1) AS total_kwh
FROM fact_energy_metering
GROUP BY airport_id, event_hour
''')
print('gold_energy_summary built')

In [ ]:
# --- gold_incidents_recent: incident feed for the report table ---
spark.sql('''
CREATE OR REPLACE TABLE gold_incidents_recent AS
SELECT incident_id, airport_id, gate_id, event_time, event_hour,
       category, severity, delay_minutes, status, description
FROM fact_operational_incidents
''')
print('gold_incidents_recent built')

In [ ]:
# Interim agent context. Advisory wording only; no operational write-back or staff/equipment control.
spark.sql('''
CREATE OR REPLACE TABLE agent_context AS
WITH gate_delay AS (
  SELECT gate_id, airport_id, MAX(departure_delay_minutes) AS max_delay,
         MAX_BY(delay_reason, departure_delay_minutes) AS worst_reason
  FROM fact_flight_turnaround_events GROUP BY gate_id, airport_id)
SELECT d.airport_id, d.gate_id,
       CASE WHEN d.max_delay > 30 THEN 'Action' WHEN d.max_delay > 15 THEN 'Watch' ELSE 'Normal' END AS operational_status,
       COALESCE(NULLIF(d.worst_reason,'None'),'None') AS delay_reason,
       CASE WHEN d.max_delay > 30 THEN 'Authorized duty manager should review the synthetic delay evidence and approve any response'
            WHEN d.max_delay > 15 THEN 'Operations lead should review the synthetic turnaround evidence'
            ELSE 'Continue monitoring; no consequential action recommended' END AS recommended_action,
       true AS advisory_only, true AS is_synthetic,
       'Derived analytical data' AS data_classification
FROM gate_delay d
''')
print('agent_context built')

In [ ]:
display(spark.table('gold_kpi_daily_summary'))
display(spark.table('agent_context'))